In [3]:
! pip install trafilatura spacy pandas httpx

In [5]:
import json
from pathlib import Path

CRAWL_JSONL_PATH = Path("crawler_output.jsonl")

print("output file:", CRAWL_JSONL_PATH.resolve())

output file: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\crawler_output.jsonl


In [7]:
#1.1
import trafilatura

SEED_URLS = [
    "https://www.un.org/en/global-issues/climate-change",
    "https://science.nasa.gov/climate-change/effects/",
    "https://www.noaa.gov/education/resource-collections/climate",
    "https://climate.nasa.gov/evidence/",
    "https://www.worldbank.org/en/topic/climatechange/overview",
    "https://unfccc.int/process-and-meetings/the-paris-agreement",
]

MIN_WORDS = 500
retained, discarded = [], []

with open(CRAWL_JSONL_PATH, "w", encoding="utf-8") as f:
    for url in SEED_URLS:
        downloaded = trafilatura.fetch_url(url)
        if downloaded is None:
            discarded.append((url, "fetch failed"))
            continue
        text = trafilatura.extract(downloaded)
        if text is None:
            discarded.append((url, "no text extracted"))
            continue
        if len(text.split()) < MIN_WORDS:
            discarded.append((url, f"too short ({len(text.split())} words)"))
            continue
        record = {"url": url, "text": text}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        retained.append((url, len(text.split())))

print(f"Retained: {len(retained)}")
for url, wc in retained:
    print(f"   {wc} words : {url}")

print(f"\n Discarded: {len(discarded)}")
for url, reason in discarded:
    print(f"   {reason} : {url}")

print(f"\nTotal words: {sum(wc for _, wc in retained)}")

Retained: 3
   1560 words : https://www.un.org/en/global-issues/climate-change
   1468 words : https://science.nasa.gov/climate-change/effects/
   1721 words : https://climate.nasa.gov/evidence/

 Discarded: 3
   fetch failed : https://www.noaa.gov/education/resource-collections/climate
   too short (451 words) : https://www.worldbank.org/en/topic/climatechange/overview
   no text extracted : https://unfccc.int/process-and-meetings/the-paris-agreement

Total words: 4749


In [9]:
import sys
print("python:", sys.version)

import pydantic
print("pydantic:", pydantic.__version__)


python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
pydantic: 2.12.5


In [11]:
import spacy
print("spacy:", spacy.__version__)


spacy: 3.8.11


In [13]:
import pydantic, sys
print("python:", sys.version)
print("pydantic:", pydantic.__version__)


python: 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]
pydantic: 2.12.5


In [16]:
!python -m spacy download en_core_web_trf

     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     -------------------------------------- 0.0/457.4 MB 445.2 kB/s eta 0:17:08
     -------------------------------------- 0.0/457.4 MB 495.5 kB/s eta 0:15:24
     -------------------------------------- 0.1/457.4 MB 550.5 kB/s eta 0:13:51
     -------------------------------------- 0.1/457.4 MB 393.8 kB/s eta 0:19:22
     -------------------------------------- 0.2/457.4 MB 798.5 kB/s eta 0:09:33
     ---------------------------------------- 0.3/457.4 MB 1.2 MB/s eta 0:06:10
     ---------------------------------------- 0.6/457.4 MB 1.9 MB/s eta 0:04:00
     ---------------------------------------- 0.6/457.4 MB 2.0 MB/s eta 0:03:45
     ---------------------------------------- 0.9/457.4 MB 2.2 MB/s eta 0:03:30
     ---------------------------------------- 0.9/457.4 MB 2.2 MB/s eta 0:03:30
     ---------------------------------------- 0.9/457.4 MB 2.2 MB/s eta 0:03:30
     ---------------------------------------- 0

In [3]:
!pip install spacy[transformers]

     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.0 kB ? eta -:--:--
     -------------------------- ----------- 30.7/44.0 kB 435.7 kB/s eta 0:00:01
     -------------------------------------- 44.0/44.0 kB 359.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/341.5 kB ? eta -:--:--
   --------------------------------------- 341.5/341.5 kB 10.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/184.3 kB ? eta -:--:--
   --------------------------------------- 184.3/184.3 kB 11.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.6/10.0 MB 11.8 MB/s eta 0:00:01
   --- ------------------------------------ 1.0/10.0 MB 10.0 MB/s eta 0:00:01
   ----- ---------------------------------- 1.3/10.0 MB 10.6 MB/s eta 0:00:01
   ------ --------------------------------- 1.7/10.0 MB 9.9 MB/s eta 0:00:01
   -------- 

In [18]:
#2.1

import spacy
nlp = spacy.load("en_core_web_trf")
print("spaCy model loaded")

spaCy model loaded


In [20]:
import csv
import re

IN_PATH = Path("crawler_output.jsonl")
OUT_PATH = Path("extracted_knowledge.csv")

wanted_labels = {"PERSON", "ORG", "GPE", "DATE"}

# pattern pour garder uniquement les dates concrètes (années, dates précises)
VALID_DATE_PATTERN = re.compile(
    r'\b\d{4}\b'                        # année seule : 2015, 2030
    r'|\b\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4}\b'  # 01/01/2020
    r'|\b(?:January|February|March|April|May|June|July|August|'
    r'September|October|November|December)\s+\d{1,2},?\s+\d{4}\b'  # January 1, 2020
)

def is_valid_date(text):
    return bool(VALID_DATE_PATTERN.search(text))

with open(IN_PATH, "r", encoding="utf-8") as f_in, open(OUT_PATH, "w", encoding="utf-8", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["entity_text", "entity_label", "source_url"])  # header

    for line in f_in:
        record = json.loads(line)
        url = record["url"]
        text = record["text"]

        doc = nlp(text) #spacy lit le texte

        for ent in doc.ents:  #doc.ents est la liste de toutes les entités trouvées
            if ent.label_ not in wanted_labels:
                continue
            # Filtre dates vagues
            if ent.label_ == "DATE" and not is_valid_date(ent.text):
                continue
            writer.writerow([ent.text, ent.label_, url]) #ent.text = le texte de l'entité (ex:Venezuela)
            #ent.label = le type (ex:PERSON)
            #writer.writerow sert à écrire une ligne dans le csv

print("csv créé:", OUT_PATH.resolve())

csv créé: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\extracted_knowledge.csv


In [22]:
#2.2

from itertools import combinations

IN_PATH = Path("crawler_output.jsonl")
OUT_PATH = Path("candidate_relations.csv")

wanted_labels = {"PERSON", "ORG", "GPE", "DATE"}

def extract_relation_token(sent):
    # relation minimale: premier verbe de la phrase, sinon "co_occurrence"
    for token in sent:
        if token.pos_ == "VERB":
            return token.lemma_
    return "co_occurrence"

with open(IN_PATH, "r", encoding="utf-8") as f_in, open(OUT_PATH, "w", encoding="utf-8", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["subject", "relation", "object", "source_url", "sentence"])

    for line in f_in:
        record = json.loads(line)
        url = record["url"]
        text = record["text"]

        doc = nlp(text)

        for sent in doc.sents:  #doc.sents spacy découpe le texte en phrases
            ents = [ent for ent in sent.ents if ent.label_ in wanted_labels] #sent.ents entités dans une phrase
            if len(ents) < 2:
                continue

            relation = extract_relation_token(sent)

            for e1, e2 in combinations(ents, 2): #toutes les paires d'entités dans la phrase
                writer.writerow([e1.text, relation, e2.text, url, sent.text])

print("relations candidates créées:", OUT_PATH.resolve())

relations candidates créées: C:\Users\rache\OneDrive - De Vinci\web datamining and semantics\td\lab1\candidate_relations.csv


In [26]:
# analyse d'ambiguïté, exemples pour le rapport

with open(IN_PATH, "r", encoding="utf-8") as f_in:
    for line in f_in:
        record = json.loads(line)
        doc = nlp(record["text"])
        for ent in doc.ents:
            if ent.label_ in wanted_labels:
                # Affiche le contexte autour de l'entité
                start = max(0, ent.start - 5)
                end = min(len(doc), ent.end + 5)
                context = doc[start:end].text
                print(f"[{ent.label_}] '{ent.text}' → contexte: ...{context}...")
                print()

[DATE] 'today' → contexte: ...scale. Without drastic action today, adapting to these impacts...

[DATE] 'the 1800s' → contexte: ...volcanic eruptions. But since the 1800s, human activities have been...

[ORG] 'the UN Environment Programme' → contexte: ...new Emissions Gas Report by the UN Environment Programme finds that there has been...

[DATE] '2015' → contexte: ...Paris Agreement was signed in 2015. Greenhouse gas emissions in...

[DATE] '2030' → contexte: .... Greenhouse gas emissions in 2030, based on policies in...

[DATE] 'Today' → contexte: ...the agreement’s adoption. Today, the projected increase is...

[DATE] '2030' → contexte: ...cent. However, predicted 2030 greenhouse gas emissions still must...

[DATE] 'over a century' → contexte: ...levels has been caused by over a century of burning fossil fuels and...

[DATE] 'this decade' → contexte: ...climate change is essential in this decade. Keeping warming to 1.5...

[DATE] '2030' → contexte: ...cut by almost half by 2030 if w